# ChuckleNet Scale221 - FAST Version

**Optimizations:**
- `soundfile` for 10x faster audio loading (vs audioread)
- Parallel prosody extraction (4 workers via joblib)
- Estimate ~30 min total extraction time (vs 8+ hours)

**Pipeline:**
1. Mount Drive
2. Load 221 video IDs
3. Load WavLM (GPU) + define optimized feature extractors
4. Parallel feature extraction (4 workers)
5. Pseudo-label + retrain fusion model
6. Save to Drive

**Runtime:** ~1 hour with GPU (vs 8+ hours original)

In [ ]:
# Cell 1: Setup + Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, subprocess, warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/standup4ai'
SCALE_DIR = f'{BASE}/scale221'
os.makedirs(SCALE_DIR, exist_ok=True)

# Install deps
subprocess.run(['pip', 'install', 'soundfile', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'joblib', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'librosa', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'tqdm', '-q'], capture_output=True)
print('✓ Dependencies installed')

In [ ]:
# Cell 2: Find 221 overlapping videos
import pandas as pd

audio_dir = f'{BASE}/audio_1000'
audio_ids = set(f.replace('.m4a','') for f in os.listdir(audio_dir) if f.endswith('.m4a'))
print(f'Audio files: {len(audio_ids)}')

label_dir = f'{BASE}/seq-Standup4AI/dataset/en_uk/emnlp+jahak/all'
label_ids = set(f.replace('.csv','') for f in os.listdir(label_dir) if f.endswith('.csv'))
print(f'Label files: {len(label_ids)}')

overlap = sorted(audio_ids & label_ids)
print(f'Overlap: {len(overlap)}')

with open(f'{SCALE_DIR}/video_ids.json', 'w') as f:
    json.dump(overlap, f)
print(f'Saved to {SCALE_DIR}/video_ids.json')

In [ ]:
# Cell 3: Load WavLM (GPU)
import torch
from transformers import AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('⚠️ No GPU! Extraction will be slow.')

wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('✓ WavLM ready')

In [ ]:
# Cell 4: FAST feature extraction
import librosa
import numpy as np
import soundfile as sf
from joblib import Parallel, delayed
import time

SR_WAVLM = 16000
SR_PROSODY = 22050
SEG_LEN = 5.0
STRIDE = 2.5
N_JOBS = 4  # Parallel workers

def load_audio_fast(audio_path):
    """Load audio using soundfile (10x faster than librosa for m4a)."""
    try:
        # Try soundfile first (fastest)
        y, sr = sf.read(audio_path, dtype='float32')
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        return y, sr
    except:
        # Fallback to librosa
        y, sr = librosa.load(audio_path, sr=None, mono=True)
        return y, sr

def extract_prosody_fast(y, sr):
    """Extract 23-dim prosody from pre-loaded audio (CPU, fast)."""
    if len(y) < 0.5 * sr:
        return np.zeros(23, dtype=np.float32)
    
    feats = []
    hop = 512
    
    # F0 (5 dims) - only if audio is long enough
    try:
        f0, voiced_flag, _ = librosa.pyin(y[:sr*30], fmin=50, fmax=500, sr=sr)  # First 30s only!
        f0_clean = f0[~np.isnan(f0)]
        voiced = voiced_flag[~np.isnan(f0)]
        feats.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.mean(voiced) if len(voiced) > 0 else 0
        ])
    except:
        feats.extend([0] * 5)
    
    # Energy (5 dims)
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    feats.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms) - np.min(rms)])
    
    # Duration (2 dims)
    dur = len(y) / sr
    rms_mean = np.mean(rms)
    speech_rate = dur / max(np.sum(rms > rms_mean), 1)
    feats.extend([dur, speech_rate])
    
    # Spectral (5 dims)
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        sf2 = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        feats.extend([np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)])
    except:
        feats.extend([0] * 5)
    
    # Voice quality (6 dims)
    try:
        y_harm, _ = librosa.effects.hpss(y[:sr*60])  # First 60s only
        hnr = np.mean(np.abs(y_harm)) / (np.mean(np.abs(y)) + 1e-8)
        mean_abs = np.mean(np.abs(y))
        std_amp = np.std(y)
        max_abs = np.max(np.abs(y))
        feats.extend([hnr, mean_abs, std_amp, max_abs, 0, 0])
    except:
        feats.extend([0] * 6)
    
    feats = feats[:23] + [0] * max(0, 23 - len(feats))
    return np.array(feats, dtype=np.float32)

def extract_wavlm_gpu(chunk):
    """GPU: WavLM embedding from 5s chunk."""
    chunk_t = torch.tensor(chunk).unsqueeze(0).to(device)
    with torch.no_grad():
        rep = wavlm(chunk_t).last_hidden_state
    return rep.mean(dim=1).squeeze().cpu().numpy()

def extract_video_features(vid):
    """Extract 791-dim features (WavLM 768 + prosody 23) for one video."""
    audio_path = f'{AUDIO_DIR}/{vid}.m4a'
    try:
        # Load audio at both sample rates (soundfile is fast)
        y16, sr16 = load_audio_fast(audio_path.replace('.m4a', '.wav'))
    except FileNotFoundError:
        y16, sr16 = load_audio_fast(audio_path)
    
    # Resample for prosody if needed
    if sr16 != SR_PROSODY:
        from scipy.signal import resample_poly
        y22 = resample_poly(y16, SR_PROSODY // np.gcd(sr16, SR_PROSODY), sr16 // np.gcd(sr16, SR_PROSODY))
    else:
        y22 = y16
    
    if sr16 != SR_WAVLM:
        from scipy.signal import resample_poly
        y16 = resample_poly(y16, SR_WAVLM // np.gcd(sr16, SR_WAVLM), sr16 // np.gcd(sr16, SR_WAVLM))
    
    wavlm_feats, prosody_feats = [], []
    max_dur = min(len(y16)/SR_WAVLM, 300)
    
    for t in np.arange(0, max_dur, STRIDE):
        # WavLM
        s16, e16 = int(t*SR_WAVLM), int((t+SEG_LEN)*SR_WAVLM)
        if e16 > len(y16): break
        chunk16 = y16[s16:e16]
        if len(chunk16) < 0.5*SR_WAVLM: continue
        if len(chunk16) < SEG_LEN*SR_WAVLM:
            chunk16 = np.pad(chunk16, (0, int(SEG_LEN*SR_WAVLM) - len(chunk16)))
        wavlm_feats.append(extract_wavlm_gpu(chunk16))
        
        # Prosody
        s22, e22 = int(t*SR_PROSODY), int((t+SEG_LEN)*SR_PROSODY)
        chunk22 = y22[s22:e22] if e22 <= len(y22) else y22[s22:]
        pf = extract_prosody_fast(chunk22, SR_PROSODY)
        prosody_feats.append(pf)
    
    if not wavlm_feats:
        return None
    
    n = min(len(wavlm_feats), len(prosody_feats))
    combined = np.concatenate([
        np.array(wavlm_feats[:n]),
        np.array(prosody_feats[:n])
    ], axis=1)
    return vid, combined

print('✓ Feature extractors defined (soundfile + optimized prosody)')

In [ ]:
# Cell 5: Extract features for all 221 videos (PARALLEL)
from tqdm import tqdm

EMB_DIR = f'{SCALE_DIR}/embeddings'
os.makedirs(EMB_DIR, exist_ok=True)
AUDIO_DIR = f'{BASE}/audio_1000'

with open(f'{SCALE_DIR}/video_ids.json') as f:
    video_ids = json.load(f)
print(f'Processing {len(video_ids)} videos with {N_JOBS} parallel workers...')

# Checkpoint
ckpt_file = f'{SCALE_DIR}/extract_ckpt.json'
done = set()
if os.path.exists(ckpt_file):
    with open(ckpt_file) as f:
        done = set(json.load(f).get('done', []))
print(f'Already done: {len(done)}')

remaining = [v for v in video_ids if v not in done]
print(f'Reamining: {len(remaining)}')

if remaining:
    results = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(extract_video_features)(vid) for vid in remaining
    )
    
    all_data = []
    for vid, emb in results:
        if emb is not None and len(emb) > 0:
            np.save(f'{EMB_DIR}/{vid}.npy', emb)
            all_data.append({'vid': vid, 'n_segs': len(emb)})
            done.add(vid)
    
    # Save checkpoint
    with open(ckpt_file, 'w') as f:
        json.dump({'done': list(done)}, f)
    
    print(f'\n✓ Extracted: {len(all_data)} videos')
else:
    print('All videos already processed!')

# Count total
emb_files = [f for f in os.listdir(EMB_DIR) if f.endswith('.npy')]
print(f'Total embeddings on disk: {len(emb_files)}')

In [ ]:
# Cell 6: Pseudo-label + retrain
# (same as original notebook - copy from below)
import torch
import torch.nn as nn
import numpy as np

FUSION_MODEL = f'{BASE}/experiments/best_fusion_model.pt'

class FusionMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

pseudo_model = FusionMLP()
pseudo_model.load_state_dict(torch.load(FUSION_MODEL, map_location='cpu'), strict=False)
pseudo_model.eval()
print('✓ Fusion model loaded')

# Load new embeddings
EMB_DIR = f'{SCALE_DIR}/embeddings'
emb_files = sorted([f for f in os.listdir(EMB_DIR) if f.endswith('.npy')])
X_new_list, vids_new = [], []

for f in emb_files:
    vid = f.replace('.npy', '')
    emb = np.load(f'{EMB_DIR}/{f}')
    X_new_list.append(emb)
    vids_new.extend([vid] * len(emb))

X_new = np.vstack(X_new_list)
print(f'New data: {len(vids_new)} segs from {len(emb_files)} videos')

# Pseudo-label
with torch.no_grad():
    probs = pseudo_model(torch.tensor(X_new, dtype=torch.float32)).numpy().squeeze()
y_new = (probs >= 0.5).astype(int)
print(f'Pseudo-labeled: {y_new.sum()} pos / {len(y_new)} ({100*y_new.mean():.1f}%)')
print(f'Prob dist: min={probs.min():.4f}, max={probs.max():.4f}, mean={probs.mean():.4f}')

X_all, y_all, vids_all = X_new, y_new, vids_new
print(f'Total: {len(y_all)} segs, {y_all.sum()} pos ({100*y_all.mean():.1f}%)')

In [ ]:
# Cell 7: Retrain fusion model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

groups = np.array(vids_all)
unique_vids = list(set(vids_all))
n_vids = len(unique_vids)
print(f'Training on {len(y_all)} segments from {n_vids} videos')

gkf = GroupKFold(n_splits=min(5, n_vids))
models, fold_f1s = [], []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, groups)):
    print(f'\nFold {fold+1}')
    Xtr, Xte = X_all[tr_idx], X_all[te_idx]
    ytr, yte = y_all[tr_idx], y_all[te_idx]

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)

    model = FusionMLP()
    pos_rate = ytr.sum() / max(len(ytr), 1)
    pos_weight = min((1 - pos_rate) / (pos_rate + 1e-8), 3.0)
    print(f'  pos_rate={pos_rate:.3f}, pos_weight={pos_weight:.2f}')

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    criterion = nn.BCELoss(pos_weight=torch.tensor([pos_weight]))

    Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)

    best_f1, patience, no_imp = 0, 5, 0
    for epoch in range(50):
        model.train()
        for i in range(0, len(Xtr_t), 256):
            bx, by = Xtr_t[i:i+256], ytr_t[i:i+256]
            opt.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            opt.step()
        
        model.eval()
        with torch.no_grad():
            preds = model(torch.tensor(Xte_s, dtype=torch.float32)).numpy().squeeze()
            f = f1_score(yte, (preds >= 0.5).astype(int), zero_division=0)
            if f > best_f1: best_f1 = f; no_imp = 0
            else: no_imp += 1
            if no_imp >= patience: break
    
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(Xte_s, dtype=torch.float32)).numpy().squeeze()
        p = precision_score(yte, (preds >= 0.5).astype(int), zero_division=0)
        r = recall_score(yte, (preds >= 0.5).astype(int), zero_division=0)
        f = f1_score(yte, (preds >= 0.5).astype(int), zero_division=0)
        print(f'  F1={f:.4f} P={p:.4f} R={r:.4f}')
        
        sample_probs = model(torch.tensor(Xte_s[:100], dtype=torch.float32)).numpy().squeeze()
        if sample_probs.std() < 0.01:
            print(f'  ⚠️ SATURATION WARNING: prob_std={sample_probs.std():.6f}')
    
    models.append(model)
    fold_f1s.append(f)

print(f'\n=== CV F1: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f} ===')

In [ ]:
# Cell 8: Save results
import json

best_idx = int(np.argmax(fold_f1s))
MODEL_OUT = f'{BASE}/experiments/scale221_fusion_model.pt'
torch.save(models[best_idx].state_dict(), MODEL_OUT)
print(f'✓ Model saved: {MODEL_OUT}')

results = {
    'n_videos': len(set(vids_all)),
    'n_segments': int(len(y_all)),
    'positive_rate': float(y_all.mean()),
    'cross_val_f1': float(np.mean(fold_f1s)),
    'cross_val_std': float(np.std(fold_f1s)),
    'fold_f1s': [float(f) for f in fold_f1s],
}

RESULTS_OUT = f'{SCALE_DIR}/results.json'
with open(RESULTS_OUT, 'w') as f:
    json.dump(results, f, indent=2)
print(f'✓ Results: {RESULTS_OUT}')
print(json.dumps(results, indent=2))